# Attention Residual Decomposition

How does attention modify the residual stream? Decompose into
parallel (reinforcing) and perpendicular (new information) components.

## Why This Matters

Composition attention residual decomposition measures how components interact across layers to implement complex computations. Understanding composition — how one head's output feeds into another's input — is essential for tracing multi-step algorithms in transformer circuits.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis
- [Olsson et al. (2022) "In-context Learning and Induction Heads"](https://arxiv.org/abs/2209.11895) — Discovers induction heads and training phase transitions

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.attention_residual_decomposition import (
    attention_parallel_perpendicular, per_head_residual_decomposition,
    attention_update_angle, attention_residual_ratio,
    attention_residual_decomposition_summary,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([1, 15, 30, 45, 60, 75, 90])
print('Model ready')

## Parallel vs Perpendicular

Parallel = reinforces current residual direction; perpendicular = adds new information.

In [ ]:
for layer in range(4):
    result = attention_parallel_perpendicular(model, tokens, layer=layer, position=-1)
    reinf = 'REINFORCING' if result['is_reinforcing'] else 'REDIRECTING'
    print(f"  Layer {layer}: par={result['parallel_magnitude']:.4f}, "
          f"perp={result['perpendicular_magnitude']:.4f}, "
          f"frac={result['parallel_fraction']:.3f} [{reinf}]")

## Per-Head Decomposition

Which heads reinforce vs redirect the residual stream?

In [ ]:
for layer in range(4):
    result = per_head_residual_decomposition(model, tokens, layer=layer, position=-1)
    print(f"  Layer {layer}: {result['n_reinforcing']}/4 reinforcing")
    for h in result['per_head']:
        flag = '+' if h['is_reinforcing'] else '-'
        print(f"    H{h['head']}: par={h['parallel']:+.4f}, "
              f"perp={h['perpendicular']:.4f} [{flag}]")

## Update Angle

Angle between attention output and residual stream at each position.

In [ ]:
for layer in range(4):
    result = attention_update_angle(model, tokens, layer=layer)
    print(f"  Layer {layer}: mean_angle={result['mean_angle']:.1f}°")
    for p in result['per_position']:
        bar = '█' * int(p['angle_degrees'] / 5)
        print(f"    Pos {p['position']}: {p['angle_degrees']:.1f}° {bar}")

## Attention/Residual Ratio

How large is attention's update relative to the residual stream?

In [ ]:
for layer in range(4):
    result = attention_residual_ratio(model, tokens, layer=layer)
    print(f"  Layer {layer}: mean ratio={result['mean_ratio']:.4f}")

## Decomposition Summary

In [ ]:
result = attention_residual_decomposition_summary(model, tokens, position=-1)
for p in result['per_layer']:
    reinf = 'REINF' if p['is_reinforcing'] else 'REDIR'
    print(f"  Layer {p['layer']}: par_frac={p['parallel_fraction']:.3f}, "
          f"angle={p['mean_angle']:.1f}° [{reinf}]")